# Native-2048 U-Net proposals + instance refiner (5 folds, pooled OOF tuning)

Inputs: competition data and the output of `charanpeeriga/filament-seg-unet2048` (prepared data, folds,
labels, 5-fold OOF and test U-Net maps). Code: `Azumah/` from branch `charan/fusion-self` (identical to main), plus
`Charan/fusion/make_unet_proposals.py`. No YOLO and no outputs from any other kernel.

1. U-Net maps -> instance proposals (components at thresholds 0.3/0.5/0.7, deduplicated) in the stage-1 format.
2. ResNet-34 refiner, 60 epochs, one per held-out fold + one on all data, U-Net channels = unet2048 maps.
3. Out-of-fold refinement of every training image, pooled 707-image PQ sweep (ranking, threshold, min area).
4. Test: 6-refiner vote, D4 TTA, pooled config.

In [ ]:
# 1. Setup
import os, subprocess, sys, json, time
from pathlib import Path

T0 = time.time()
REPO = Path('/tmp/aml_project')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'charan/fusion-self',
                    'https://github.com/hsfcoulibaly/aml_project.git', str(REPO)], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
CODE = Path('/kaggle/working/filament-seg')
if not CODE.exists():
    subprocess.run(['cp', '-r', str(REPO / 'Azumah'), str(CODE)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch', 'timm',
                'pycocotools'], check=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.chdir(CODE)
sys.path.insert(0, str(CODE))
os.makedirs('logs', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Link the U-Net-2048 run into one work dir
WORK = Path('/kaggle/working/fseg')
(WORK / 'unet' / 'maps').mkdir(parents=True, exist_ok=True)
inp = Path('/kaggle/input')
base = next(p.parent for p in inp.rglob('manifest.csv') if (p.parent / 'references.pkl').exists())
for name in ['manifest.csv', 'targets.pkl', 'references.pkl', 'preprocessed']:
    if not (WORK / name).exists():
        (WORK / name).symlink_to(base / name)
maps = next(p for p in inp.rglob('maps/unet2048') if (p / 'oof').exists() and (p / 'test').exists())
if not (WORK / 'unet' / 'maps' / 'unet2048').exists():
    (WORK / 'unet' / 'maps' / 'unet2048').symlink_to(maps)
os.environ['FSEG_WORK_DIR'] = str(WORK)
print('base', base, '| maps', maps, len(list((maps / 'oof').glob('*.png'))), len(list((maps / 'test').glob('*.png'))))

In [ ]:
# 3. Proposals from the U-Net maps (CPU, ~10 min)
!python {REPO}/Charan/fusion/make_unet_proposals.py --code {CODE} --maps {WORK}/unet/maps/unet2048 --workers 4
print(f'elapsed {(time.time() - T0) / 3600:.2f} h')

In [ ]:
# 4. Train 6 refiners (folds 0-4 held out + full data), three per GPU in sequence
JOBS = {0: ['0', '1', '2'], 1: ['3', '4', 'none']}
name = lambda k: 'u2r_full' if k == 'none' else f'u2r_v{k}'
procs = []
for gpu, folds in JOBS.items():
    chain = '; '.join(f'python scripts/train_refiner.py --name {name(k)} --val-fold {k} --unet-name unet2048 '
                      f'--arch unet --epochs 60 --workers 2 --max-matched 12 --max-neg 8 --resume '
                      f'> logs/{name(k)}.log 2>&1' for k in folds)
    procs.append(subprocess.Popen(f'CUDA_VISIBLE_DEVICES={gpu} bash -c "{chain}"', shell=True))
for p in procs:
    p.wait()
for k in ['0', '1', '2', '3', '4', 'none']:
    print(k, open(f'logs/{name(k)}.log').read().strip().splitlines()[-1])
print(f'elapsed {(time.time() - T0) / 3600:.2f} h')

In [ ]:
# 5. Out-of-fold refinement of all 707 images (each fold by the refiner that never saw it)
JOBS = {0: [0, 1, 2], 1: [3, 4]}
procs = []
for gpu, folds in JOBS.items():
    chain = '; '.join(f'python scripts/predict_refiner.py oof --fold {k} --refiner u2r_v{k} --tag u2r_f{k} '
                      f'--unet-name unet2048 --tta d4 > logs/oof_f{k}.log 2>&1' for k in folds)
    procs.append(subprocess.Popen(f'CUDA_VISIBLE_DEVICES={gpu} bash -c "{chain}"', shell=True))
for p in procs:
    p.wait()
for k in range(5):
    print(k, open(f'logs/oof_f{k}.log').read().strip().splitlines()[-1][:250])
print(f'elapsed {(time.time() - T0) / 3600:.2f} h')

In [ ]:
# 6. Pooled 707-image OOF PQ ('yolo' column = U-Net-2048 alone, thr .5 / area 400 / mean .7)
!python scripts/predict_refiner.py pool --tag u2r --folds 0 1 2 3 4 --tta d4

In [ ]:
# 7. Test: 6-refiner vote with the pooled config, and the full-data refiner alone
P = WORK / 'refiner' / 'u2r_pooled'
cfg = P / 'pq_config.json'
raw = WORK / 'runs' / 'unet2048_test' / 'test_raw.pkl'
six = ' '.join([f'u2r_v{k}' for k in range(5)] + ['u2r_full'])
!python scripts/predict_refiner.py test --refiner {six} --config {cfg} --raw {raw} --unet-name unet2048 --tag test_u2rx6 --out /kaggle/working/submission_u2rx6_pooled.csv
!python scripts/predict_refiner.py test --refiner u2r_full --config {cfg} --raw {raw} --unet-name unet2048 --tag test_u2rfull --out /kaggle/working/submission_u2rfull_pooled.csv
print(open(cfg).read())
print(open(P / 'per_fold.csv').read())
print(f'total {(time.time() - T0) / 3600:.2f} h')

In [ ]:
# 8. Keep the output small: drop symlinks and crop caches, keep weights, pickles, logs, CSVs
import shutil
for name in ['manifest.csv', 'targets.pkl', 'references.pkl', 'preprocessed']:
    (WORK / name).unlink(missing_ok=True)
(WORK / 'unet' / 'maps' / 'unet2048').unlink(missing_ok=True)
for f in WORK.rglob('last_state.pt'):
    f.unlink()
!du -sh /kaggle/working/* ; ls -la /kaggle/working/*.csv